# EuroSAT Visualization and Sentinel-2 Band Exploration

This notebook demonstrates how to load images from the EuroSAT dataset, visualize samples from various land cover classes, and explore the different spectral bands provided by the Sentinel-2 mission.

## Sentinel-2 Band Explanations

Sentinel-2 captures images in 13 spectral bands. For this dataset, we specifically highlight the following essential bands:

- **Band 2 (Blue - 490 nm):** Useful for water body discrimination, soil/vegetation discrimination, and forest type mapping.
- **Band 3 (Green - 560 nm):** Useful for estimating vegetation peak reflectance to assess plant vigor.
- **Band 4 (Red - 665 nm):** Located in the chlorophyll absorption band, making it important for determining vegetation types and health.
- **Band 8 (Near-Infrared / NIR - 842 nm):** Healthy vegetation reflects highly in the NIR. This band is crucial for calculating indices like NDVI (Normalized Difference Vegetation Index).
- **Band 11 (Short Wave Infrared / SWIR - 1610 nm):** Highly sensitive to moisture content in soil and vegetation. It is extremely useful for drought monitoring, agriculture, and differentiating between snow and clouds.

In [ ]:
import os
import logging
from typing import Dict, Optional
from glob import glob

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.errors import RasterioIOError

# Configure basic logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger()

In [ ]:
# Define paths assuming the EuroSAT dataset has been downloaded and extracted
DATA_DIR = '../data/raw/eurosat/extracted/ds/images/remote_sensing/otherDatasets/sentinel_2/tif'

def get_sample_per_class(base_dir: str) -> Dict[str, str]:
    """
    Retrieve one sample image file path for each class in the dataset.
    
    Args:
        base_dir (str): The root directory containing class subdirectories.
        
    Returns:
        Dict[str, str]: A dictionary mapping class names to sample image file paths.
    """
    if not os.path.exists(base_dir):
        logger.error("Directory %s not found. Please run the download script first.", base_dir)
        return {}
        
    classes = os.listdir(base_dir)
    samples = {}
    for class_name in classes:
        class_dir = os.path.join(base_dir, class_name)
        if os.path.isdir(class_dir):
            files = glob(os.path.join(class_dir, '*.tif'))
            if files:
                samples[class_name] = files[0]
    return samples

sample_files = get_sample_per_class(DATA_DIR)
if sample_files:
    logger.info("Found samples for %d classes.", len(sample_files))

## Visualizing Samples from Every Class
We will load the RGB bands (Band 4, Band 3, Band 2) to visualize one sample from each class. Remote sensing images often need basic stretching to be displayed clearly in standard 8-bit RGB color space.

In [ ]:
def load_rgb(file_path: str) -> Optional[np.ndarray]:
    """
    Load the RGB bands of a Sentinel-2 multi-spectral image and normalize for visualization.
    
    Args:
        file_path (str): The path to the TIFF image.
        
    Returns:
        Optional[np.ndarray]: A 3D numpy array representing the normalized RGB image, or None if failed.
    """
    try:
        with rasterio.open(file_path) as src:
            # EuroSAT multi-spectral bands order: B01, B02(Blue), B03(Green), B04(Red), ...
            red_band = src.read(4)
            green_band = src.read(3)
            blue_band = src.read(2)
            
            # Stack bands to create an RGB image
            rgb_image = np.dstack((red_band, green_band, blue_band))
            
            # Normalize and apply a simple contrast stretch (multiplier 2.5)
            max_val = np.max(rgb_image)
            if max_val > 0:
                rgb_image = (rgb_image / max_val) * 2.5
            
            # Clip values to valid [0, 1] range for matplotlib
            rgb_image = np.clip(rgb_image, 0, 1)
            return rgb_image
    except RasterioIOError as e:
        logger.error("Failed to open image %s: %s", file_path, e)
        return None

if sample_files:
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()
    
    for ax, (cls_name, file_path) in zip(axes, sample_files.items()):
        img_rgb = load_rgb(file_path)
        if img_rgb is not None:
            ax.imshow(img_rgb)
        ax.set_title(cls_name)
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    logger.warning("No samples to display.")

## RGB vs NIR Visualization
Here we compare a true-color (RGB) composite with a false-color Near-Infrared (NIR) composite. In the NIR composite, vegetation appears bright red because healthy plant life reflects heavily in the Near-Infrared spectrum.

In [ ]:
def load_nir_false_color(file_path: str) -> Optional[np.ndarray]:
    """
    Load a false-color NIR composite of a Sentinel-2 image.
    
    Args:
        file_path (str): The path to the TIFF image.
        
    Returns:
        Optional[np.ndarray]: A 3D numpy array representing the normalized false-color image, or None if failed.
    """
    try:
        with rasterio.open(file_path) as src:
            # False color NIR mapping: Red channel = NIR (B08), Green channel = Red (B04), Blue channel = Green (B03)
            nir_band = src.read(8)
            red_band = src.read(4)
            green_band = src.read(3)
            
            false_color_img = np.dstack((nir_band, red_band, green_band))
            
            max_val = np.max(false_color_img)
            if max_val > 0:
                false_color_img = (false_color_img / max_val) * 2.5
                
            false_color_img = np.clip(false_color_img, 0, 1)
            return false_color_img
    except RasterioIOError as e:
        logger.error("Failed to open image %s: %s", file_path, e)
        return None

if sample_files:
    # Select the first sample from the dictionary for comparison
    sample_class = list(sample_files.keys())[0]
    sample_file = sample_files[sample_class]
    
    img_rgb = load_rgb(sample_file)
    img_nir = load_nir_false_color(sample_file)
    
    if img_rgb is not None and img_nir is not None:
        fig, axes = plt.subplots(1, 2, figsize=(14, 7))
        
        axes[0].imshow(img_rgb)
        axes[0].set_title(f'RGB True-Color ({sample_class})')
        axes[0].axis('off')
        
        axes[1].imshow(img_nir)
        axes[1].set_title(f'NIR False-Color ({sample_class})\n(Red channel = NIR band)')
        axes[1].axis('off')
        
        plt.show()
    else:
        logger.error("Failed to load images for visualization.")